In [13]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import numpy as np
import pandas as pd

from src.evaluation import evaluate_recommender, ndcg_at_k, precision_at_k, recall_at_k
from src.matrix_factorization import MatrixFactorization
from src.preprocessing import leave_k_last

In [14]:
input_dir = Path("../data/processed/")

train = pd.read_csv(input_dir / "train.csv")
test = pd.read_csv(input_dir / "test.csv")

In [15]:
unique_users = train["userId"].unique()
unique_items = train["movieId"].unique()
n_users = len(unique_users)
n_items = len(unique_items)

k = 100
mu = train["rating"].mean()
bias_user = np.zeros(n_users)
bias_item = np.zeros(n_items)
p_matrix = np.random.normal(0, 0.01, size=(n_users, k))
q_matrix = np.random.normal(0, 0.01, size=(n_items, k))

user_id_to_idx = {ids: idx for idx, ids in enumerate(unique_users)}
item_id_to_idx = {ids: idx for idx, ids in enumerate(unique_items)}

In [16]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:

    return np.sqrt(np.mean((y_true - y_pred) ** 2))

In [17]:
train_fit, val_fit = leave_k_last(train, k=1)

In [18]:
epochs = 100
learning_rate = 0.01
lm = 0.01

best_val_loss = float("inf")
best_epoch = 0
patience_counter = 0
patience = 10

p_matrix_best = p_matrix.copy()
q_matrix_best = q_matrix.copy()
bias_user_best = bias_user.copy()
bias_item_best = bias_item.copy()

for epoch in range(epochs):
    shuffled_train = train_fit.sample(frac=1, random_state=42)

    train_loss = 0.0
    squared_error = 0.0
    for row in shuffled_train.itertuples():
        u = user_id_to_idx[row.userId]
        i = item_id_to_idx[row.movieId]
        r = row.rating

        rating_pred = mu + bias_user[u] + bias_item[i] + p_matrix[u] @ q_matrix[i]
        error = r - rating_pred

        p_old = p_matrix[u].copy()
        q_old = q_matrix[i].copy()
        p_matrix[u] += learning_rate * (error * q_old - lm * p_old)
        q_matrix[i] += learning_rate * (error * p_old - lm * q_old)
        bias_user[u] += learning_rate * (error - lm * bias_user[u])
        bias_item[i] += learning_rate * (error - lm * bias_item[i])

        squared_error += error**2

    train_loss = np.sqrt(squared_error / len(shuffled_train))

    squared_error = 0.0
    val_loss = 0.0
    for row in val_fit.itertuples():
        u = user_id_to_idx[row.userId]
        i = item_id_to_idx[row.movieId]
        r = row.rating

        rating_pred = mu + bias_user[u] + bias_item[i] + p_matrix[u] @ q_matrix[i]
        squared_error += (r - rating_pred) ** 2

    val_loss = np.sqrt(squared_error / len(val_fit))

    print(f"[{epoch}/{epochs}] Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        p_matrix_best = p_matrix.copy()
        q_matrix_best = q_matrix.copy()
        bias_user_best = bias_user.copy()
        bias_item_best = bias_item.copy()

        patience_counter = 0

    else:
        patience_counter += 1

        if patience_counter == patience:
            break

[0/100] Train loss: 0.9355, Val loss: 0.9316
[1/100] Train loss: 0.8873, Val loss: 0.9032
[2/100] Train loss: 0.8709, Val loss: 0.8917
[3/100] Train loss: 0.8606, Val loss: 0.8860
[4/100] Train loss: 0.8531, Val loss: 0.8831
[5/100] Train loss: 0.8471, Val loss: 0.8816
[6/100] Train loss: 0.8419, Val loss: 0.8808
[7/100] Train loss: 0.8371, Val loss: 0.8805
[8/100] Train loss: 0.8320, Val loss: 0.8804
[9/100] Train loss: 0.8261, Val loss: 0.8804
[10/100] Train loss: 0.8188, Val loss: 0.8803
[11/100] Train loss: 0.8093, Val loss: 0.8801
[12/100] Train loss: 0.7970, Val loss: 0.8797
[13/100] Train loss: 0.7818, Val loss: 0.8788
[14/100] Train loss: 0.7638, Val loss: 0.8775
[15/100] Train loss: 0.7436, Val loss: 0.8758
[16/100] Train loss: 0.7215, Val loss: 0.8738
[17/100] Train loss: 0.6983, Val loss: 0.8716
[18/100] Train loss: 0.6745, Val loss: 0.8695
[19/100] Train loss: 0.6504, Val loss: 0.8677
[20/100] Train loss: 0.6264, Val loss: 0.8662
[21/100] Train loss: 0.6027, Val loss: 0.865

In [19]:
p_matrix = p_matrix_best
q_matrix = q_matrix_best
bias_user = bias_user_best
bias_item = bias_item_best

In [20]:
train_bpr = train[train["rating"] >= 3.5].copy()
train_fit, val_fit = leave_k_last(train_bpr, k=1)
model = MatrixFactorization(
    train=train_fit, val=val_fit, learning_rate=0.01, lm=0.01, k=20, random_state=142
)
model.fit(1)

[01/100] Train BPR Loss: 0.7053 | Val BPR Loss: 0.7041
[02/100] Train BPR Loss: 0.6986 | Val BPR Loss: 0.7106
[03/100] Train BPR Loss: 0.6934 | Val BPR Loss: 0.7105
[04/100] Train BPR Loss: 0.6871 | Val BPR Loss: 0.7004
[05/100] Train BPR Loss: 0.6789 | Val BPR Loss: 0.6905
[06/100] Train BPR Loss: 0.6653 | Val BPR Loss: 0.6980
[07/100] Train BPR Loss: 0.6428 | Val BPR Loss: 0.6830
[08/100] Train BPR Loss: 0.6080 | Val BPR Loss: 0.6583
[09/100] Train BPR Loss: 0.5640 | Val BPR Loss: 0.6458
[10/100] Train BPR Loss: 0.5179 | Val BPR Loss: 0.6078
[11/100] Train BPR Loss: 0.4746 | Val BPR Loss: 0.5914
[12/100] Train BPR Loss: 0.4390 | Val BPR Loss: 0.5505
[13/100] Train BPR Loss: 0.4104 | Val BPR Loss: 0.5366
[14/100] Train BPR Loss: 0.3879 | Val BPR Loss: 0.5100
[15/100] Train BPR Loss: 0.3689 | Val BPR Loss: 0.4785
[16/100] Train BPR Loss: 0.3564 | Val BPR Loss: 0.4638
[17/100] Train BPR Loss: 0.3428 | Val BPR Loss: 0.4460
[18/100] Train BPR Loss: 0.3307 | Val BPR Loss: 0.4298
[19/100] T

In [21]:
test["predict"] = test["userId"].apply(lambda uid: model.recommend_top_n(uid, n=10))

In [22]:
test_pos = test[test["rating"] >= 3.5]
eval_df = test_pos.groupby("userId")["movieId"].apply(set).reset_index(name="relevant")


eval_df = eval_df[eval_df["userId"].isin(model.user_id_to_idx)].copy()


k = 10
eval_df["predict"] = eval_df["userId"].apply(
    lambda uid: [movie_id for movie_id, _ in model.recommend_top_n(uid, n=k)]
)

eval_df[f"precision@{k}"] = eval_df.apply(
    lambda row: precision_at_k(row["predict"], row["relevant"], k), axis=1
)

eval_df[f"recall@{k}"] = eval_df.apply(
    lambda row: recall_at_k(row["predict"], row["relevant"], k), axis=1
)

eval_df[f"ndcg@{k}"] = eval_df.apply(
    lambda row: ndcg_at_k(row["predict"], row["relevant"], k), axis=1
)

mean_precision = eval_df[f"precision@{k}"].mean()
mean_recall = eval_df[f"recall@{k}"].mean()
mean_ndcg = eval_df[f"ndcg@{k}"].mean()

print(f"Mean Precision@{k}: {mean_precision:.4f}")
print(f"Mean Recall@{k}:    {mean_recall:.4f}")
print(f"Mean nDCG@{k}:      {mean_ndcg:.4f}")

Mean Precision@10: 0.0071
Mean Recall@10:    0.0709
Mean nDCG@10:      0.0397


In [23]:
model = MatrixFactorization(train_fit, val_fit, learning_rate=0.01, lm=0.01, random_state=142)
model.fit(1)
results = evaluate_recommender(model=model, test_df=test, k=10, rating_threshold=3.5)

print("--- Test Evaluation Results ---")
for metric_name, score in results.items():
    print(f"{metric_name}: {score:.4f}")

[01/100] Train BPR Loss: 0.7053 | Val BPR Loss: 0.7041
[02/100] Train BPR Loss: 0.6986 | Val BPR Loss: 0.7106
[03/100] Train BPR Loss: 0.6934 | Val BPR Loss: 0.7105
[04/100] Train BPR Loss: 0.6871 | Val BPR Loss: 0.7004
[05/100] Train BPR Loss: 0.6789 | Val BPR Loss: 0.6905
[06/100] Train BPR Loss: 0.6653 | Val BPR Loss: 0.6980
[07/100] Train BPR Loss: 0.6428 | Val BPR Loss: 0.6830
[08/100] Train BPR Loss: 0.6080 | Val BPR Loss: 0.6583
[09/100] Train BPR Loss: 0.5640 | Val BPR Loss: 0.6458
[10/100] Train BPR Loss: 0.5179 | Val BPR Loss: 0.6078
[11/100] Train BPR Loss: 0.4746 | Val BPR Loss: 0.5914
[12/100] Train BPR Loss: 0.4390 | Val BPR Loss: 0.5505
[13/100] Train BPR Loss: 0.4104 | Val BPR Loss: 0.5366
[14/100] Train BPR Loss: 0.3879 | Val BPR Loss: 0.5100
[15/100] Train BPR Loss: 0.3689 | Val BPR Loss: 0.4785
[16/100] Train BPR Loss: 0.3564 | Val BPR Loss: 0.4638
[17/100] Train BPR Loss: 0.3428 | Val BPR Loss: 0.4460
[18/100] Train BPR Loss: 0.3307 | Val BPR Loss: 0.4298
[19/100] T